In [1]:
print("rrre")

rrre


In [2]:
import os

os.environ["OPENAI_API_KEY"] = "ollama"
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_MODEL"] = "qwen2.5:3b"


In [3]:

from __future__ import annotations

import argparse
import base64
import json
import os
import re
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


In [4]:

import pandas as pd

from lida import Manager, llm


In [5]:
from openai import OpenAI


In [ ]:
def flatten_json(obj: Any, parent: str = "", sep: str = ".", max_depth: int = 12, _d: int = 0) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    if _d > max_depth:
        out[parent or "value"] = json.dumps(obj, ensure_ascii=False)
        return out
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{parent}{sep}{k}" if parent else str(k)
            if isinstance(v, dict):
                out.update(flatten_json(v, key, sep, max_depth, _d + 1))
            elif isinstance(v, list):
                out[key] = json.dumps(v, ensure_ascii=False)
            else:
                out[key] = v
        return out
    if isinstance(obj, list):
        out[parent or "value"] = json.dumps(obj, ensure_ascii=False)
        return out
    out[parent or "value"] = obj
    return out


def read_ndjson_rows(path: Path, limit: int = 5000) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                rows.append(flatten_json(obj))
    return rows


def read_json_file_rows(path: Path, limit: int = 5000) -> List[Dict[str, Any]]:
    txt = path.read_text(encoding="utf-8").strip()
    if not txt:
        return []
    if txt.startswith("["):
        data = json.loads(txt)
        if not isinstance(data, list):
            data = [data]
        data = data[:limit]
        return [flatten_json(x) if isinstance(x, dict) else {"value": json.dumps(x, ensure_ascii=False)} for x in data]
    return read_ndjson_rows(path, limit=limit)


def load_any_df(path: str, sample_rows: int = 5000) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    ext = p.suffix.lower()
    if ext in [".csv", ".tsv"]:
        sep = "\t" if ext == ".tsv" else ","
        try:
            return pd.read_csv(p, sep=sep, nrows=sample_rows, low_memory=False)
        except Exception:
            return pd.read_csv(p, sep=None, engine="python", nrows=sample_rows, low_memory=False)
    if ext in [".json", ".ndjson", ".jsonl", ".log"]:
        return pd.DataFrame(read_json_file_rows(p, limit=sample_rows))
    return pd.read_csv(p, nrows=sample_rows, low_memory=False)


def coerce_basic_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == "object":
            s = out[col].dropna().astype(str).head(80)
            if s.empty:
                continue
            numeric_like = s.str.match(r"^\s*-?\d+(\.\d+)?\s*$").mean()
            if numeric_like >= 0.85:
                out[col] = pd.to_numeric(out[col], errors="coerce")
    return out




def summarize_columns(df: pd.DataFrame, max_cols: int = 120) -> List[Dict[str, Any]]:
    """
    Build a compact list: name, dtype, non-null ratio, n_unique, examples, numeric range (if numeric).
    """
    cols = list(map(str, df.columns))
    profs: List[Tuple[float, str, Dict[str, Any]]] = []

    n = len(df) if len(df) else 1

    for c in cols:
        s = df[c]
        non_null = float(s.notna().mean())
        nunique = int(s.nunique(dropna=True))
        dtype = str(s.dtype)

        ex_vals = s.dropna().head(5).tolist()
        examples = [str(v)[:120] for v in ex_vals]

        cinfo: Dict[str, Any] = {
            "name": c,
            "dtype": dtype,
            "non_null": round(non_null, 3),
            "n_unique": nunique,
            "examples": examples,
        }

        if pd.api.types.is_numeric_dtype(s):
            cmin = pd.to_numeric(s, errors="coerce").min()
            cmax = pd.to_numeric(s, errors="coerce").max()
            if pd.notna(cmin) and pd.notna(cmax):
                cinfo["range"] = [float(cmin), float(cmax)]

        score = 2.0 * non_null
        if nunique <= 1:
            score -= 3.0
        elif 2 <= nunique <= 30:
            score += 0.8
        else:
            score += 0.2
        if "float" in dtype or "int" in dtype:
            score += 1.0
        if re.search(r"(time|timestamp|date|simtime)", c, re.I):
            score += 0.8
        if re.search(r"(raw|payload|message|original)", c, re.I):
            score -= 0.5

        profs.append((score, c, cinfo))

    profs.sort(reverse=True, key=lambda t: t[0])
    return [info for _, _, info in profs[:max_cols]]


#Mapper colonne avec llm

MAPPER_SYSTEM = """Return ONLY a single JSON object. No markdown. No explanation. No extra text.

Schema:
{
  "x": string|null,
  "y": string|null,
  "color": string|null,
  "size": string|null,
  "facet": string|null,
  "time": string|null,
  "filter": [{"column": string, "op": "=="|"!="|">"|">="|"<"|"<="|"in"|"contains", "value": any}],
  "chart": "scatter"|"line"|"bar"|"hist"|"heatmap"|"box"|"auto",
  "reasoning_brief": string
}

Rules:
- Use only column names that appear in SCHEMA.
- If unsure, set fields to null and chart to "auto".
- reasoning_brief must be <= 120 chars.
"""


def llm_map_columns(user_prompt: str, schema: List[Dict[str, Any]]) -> Dict[str, Any]:
    # --- OLLAMA LOCAL ---
    client = OpenAI(
        api_key="ollama",                      # valeur quelconque
        base_url="http://localhost:11434/v1"   # API locale Ollama
    )
    model = "qwen2.5:3b"
    # -------------------

    schema_text = json.dumps(schema, ensure_ascii=False)
    msg = f"SCHEMA:\n{schema_text}\n\nUSER_PROMPT:\n{user_prompt}\n\nReturn JSON only."

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": MAPPER_SYSTEM},
            {"role": "user", "content": msg},
        ],
        temperature=0.1,
    )

    content = (resp.choices[0].message.content or "").strip()

    try:
        mapped = parse_json_object_loose(content)
    except Exception:
        # fallback minimal si le modèle ne respecte pas le format
        mapped = {
            "x": None, "y": None, "color": None, "size": None, "facet": None, "time": None,
             "filter": [],
             "chart": "auto",
            "reasoning_brief": "fallback: model did not return valid JSON"
        }

    return mapped



# ---------------------------
# 4) Build LIDA goal from mapping (no manual columns)
# ---------------------------

def build_goal_from_mapping(user_prompt: str, mapping: Dict[str, Any]) -> Dict[str, Any]:
    chart = (mapping.get("chart") or "auto").lower()
    x = mapping.get("x")
    y = mapping.get("y")
    color = mapping.get("color")
    time_col = mapping.get("time")
    filters = mapping.get("filter") or []

    # Une instruction "visualization" claire pour LIDA
    viz_parts = []

    if chart == "auto":
        viz_parts.append("Create the most appropriate visualization for the request using matplotlib.")
    else:
        viz_parts.append(f"Create a {chart} chart using matplotlib.")

    if time_col:
        viz_parts.append(f"If a time axis is relevant, use '{time_col}' as time.")

    if x and y:
        viz_parts.append(f"Use x='{x}' and y='{y}'.")
    elif x:
        viz_parts.append(f"Use x='{x}'.")

    if color:
        viz_parts.append(f"Use '{color}' for color/grouping if helpful.")

    if filters:
        viz_parts.append("Apply filters:")
        for flt in filters:
            viz_parts.append(f"- {flt['column']} {flt['op']} {flt['value']}")

    visualization = "\n".join(viz_parts)

    rationale = (mapping.get("reasoning_brief") or "Mapped prompt to columns and chart type.").strip()
    if len(rationale) > 200:
        rationale = rationale[:200]

    # IMPORTANT: LIDA Goal attend ces 3 champs
    return {
        "question": user_prompt.strip(),
        "visualization": visualization,
        "rationale": rationale,
    }




def reduce_df_to_mapping(df: pd.DataFrame, mapping: Dict[str, Any], max_extra_numeric: int = 3) -> pd.DataFrame:
    """
    LIDA works best with few columns. Keep mapped columns + a few extra numeric columns for fallback.
    """
    wanted = []
    for k in ["x", "y", "color", "size", "facet", "time"]:
        c = mapping.get(k)
        if isinstance(c, str) and c in df.columns:
            wanted.append(c)

   
    for flt in (mapping.get("filter") or []):
        col = flt.get("column")
        if isinstance(col, str) and col in df.columns:
            wanted.append(col)

    wanted = list(dict.fromkeys(wanted))  

    if len(wanted) < 6:
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        scored = []
        for c in numeric_cols:
            s = df[c]
            nn = float(s.notna().mean())
            nu = int(s.nunique(dropna=True))
            if nn < 0.2 or nu <= 1:
                continue
            scored.append((2.0 * nn, c))
        scored.sort(reverse=True, key=lambda t: t[0])
        for _, c in scored[:max_extra_numeric]:
            if c not in wanted:
                wanted.append(c)

    if not wanted:
        return df.head(200)  # fallback
    return df[wanted].copy()


def apply_filters(df: pd.DataFrame, mapping: Dict[str, Any]) -> pd.DataFrame:
    filters = mapping.get("filter") or []
    out = df
    for flt in filters:
        col = flt.get("column")
        op = flt.get("op")
        val = flt.get("value")
        if col not in out.columns:
            continue
        s = out[col]

        if op == "==":
            out = out[s == val]
        elif op == "!=":
            out = out[s != val]
        elif op == ">":
            out = out[pd.to_numeric(s, errors="coerce") > float(val)]
        elif op == ">=":
            out = out[pd.to_numeric(s, errors="coerce") >= float(val)]
        elif op == "<":
            out = out[pd.to_numeric(s, errors="coerce") < float(val)]
        elif op == "<=":
            out = out[pd.to_numeric(s, errors="coerce") <= float(val)]
        elif op == "contains":
            out = out[s.astype(str).str.contains(str(val), na=False)]
        elif op == "in":
            if isinstance(val, list):
                out = out[s.isin(val)]
    return out


def save_chart_png(chart_obj: Any, out_path: Path) -> None:
    raster = getattr(chart_obj, "raster", None)
    if raster is None:
        raise RuntimeError("Chart has no raster attribute.")
    if isinstance(raster, (bytes, bytearray)):
        out_path.write_bytes(bytes(raster))
        return
    if isinstance(raster, str):
        if raster.startswith("data:image"):
            raster = raster.split(",", 1)[-1]
        out_path.write_bytes(base64.b64decode(raster))
        return
    raise RuntimeError(f"Unsupported raster type: {type(raster)}")



# Main


def run_lida(
    file: str,
    prompt: str,
    out: str = "out.png",
    sample_rows: int = 5000,
    schema_cols: int = 120,
    n_goals: int = 6,
):
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Missing OPENAI_API_KEY env var")

    df = load_any_df(file, sample_rows=sample_rows)
    if df.empty:
        raise RuntimeError("Empty dataframe (parsed but no rows).")
    df = coerce_basic_types(df)


    schema = summarize_columns(df, max_cols=schema_cols)


    mapping = llm_map_columns(prompt, schema)
    mapping = sanitize_mapping(mapping, df)

    print("MAPPING (sanitized):\n", json.dumps(mapping, ensure_ascii=False, indent=2))



    df2 = apply_filters(df, mapping)
    df_small = reduce_df_to_mapping(df2, mapping)


    import tempfile
    from pathlib import Path

    with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False, encoding="utf-8", newline="") as tmp:
        tmp_path = Path(tmp.name)
        df_small.to_csv(tmp, index=False)

    try:
        lida_mgr = Manager(text_gen=llm("openai", model="qwen2.5:3b"))
        summary = lida_mgr.summarize(str(tmp_path))

        goal = build_goal_from_mapping(prompt, mapping)
        charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")

        if not charts:
            raise RuntimeError("No charts returned by LIDA.")

        save_chart_png(charts[0], Path(out))

        code = getattr(charts[0], "code", None)
        if isinstance(code, str) and code.strip():
            Path(out).with_suffix(".generated.py").write_text(code, encoding="utf-8")

        print(f" Saved: {out}")

    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass


In [ ]:
def parse_json_object_loose(text: str) -> dict:
    """
    Try hard to extract a JSON object from LLM output:
    - removes ```json fences
    - finds first {...} block
    - ensures it's a dict
    """
    import json, re

    t = (text or "").strip()

  
    t = re.sub(r"^```(?:json)?\s*", "", t.strip(), flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t.strip())

 
    try:
        obj = json.loads(t)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

  
    start = t.find("{")
    end = t.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = t[start:end+1]
        candidate = candidate.replace("\u201c", '"').replace("\u201d", '"')  # smart quotes
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    raise ValueError("Could not extract a JSON object from model output")


In [ ]:
def sanitize_mapping(mapping: dict, df: pd.DataFrame) -> dict:
    cols = set(df.columns)

    for k in ["x", "y", "color", "size", "facet", "time"]:
        v = mapping.get(k)
        if isinstance(v, str) and v not in cols:
            mapping[k] = None

    
    valid_filters = []
    for f in mapping.get("filter", []):
        if (
            isinstance(f, dict)
            and f.get("column") in cols
            and f.get("op") in ["==","!=",">",">=","<","<=","in","contains"]
        ):
            valid_filters.append(f)
    mapping["filter"] = valid_filters

    return mapping       

In [9]:
run_lida(
    file="data/lidata.log",
    prompt="Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap",
    out="out.png",
    schema_cols=25
)


MAPPING (sanitized):
 {
  "x": "ForceIdentifier",
  "y": null,
  "color": null,
  "size": null,
  "facet": null,
  "time": null,
  "filter": [
    {
      "column": "HasAmmunitionSupplyCap",
      "op": "==",
      "value": true
    }
  ],
  "chart": "pie",
  "reasoning_brief": "To create a pie chart, we need to group by 'ForceIdentifier' and count the number of entities that have HasAmmunitionSupplyCap set to True."
}
 Saved: out.png


<Figure size 1000x600 with 0 Axes>

In [10]:
# Correction automatique du mapping pour garantir l'utilisation de la colonne demandée

def correct_mapping_with_prompt(mapping, schema, prompt):
    schema_cols = [col['name'] for col in schema]
    for col in schema_cols:
        if col in prompt:
            for key in ['x', 'y', 'color', 'facet', 'size', 'time']:
                if mapping.get(key) != col and col in schema_cols:
                    mapping[key] = col
    return mapping


In [13]:
prompt = "Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap"
schema = summarize_columns(df, max_cols=schema_cols)
mapping = llm_map_columns(prompt, schema)
mapping = correct_mapping_with_prompt(mapping, schema, prompt)

NameError: name 'df' is not defined

In [15]:
# Chargement du dataframe et correction du mapping
file = "data/lidata.log"
sample_rows = 5000
schema_cols = 25
prompt = "Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l'axe x et HasAmmunitionSupplyCap pour l'axe y."

df = load_any_df(file, sample_rows=sample_rows)
df = coerce_basic_types(df)
schema = summarize_columns(df, max_cols=schema_cols)
mapping = llm_map_columns(prompt, schema)
mapping = sanitize_mapping(mapping, df)
mapping = correct_mapping_with_prompt(mapping, schema, prompt)
print("MAPPING (corrigé):\n", json.dumps(mapping, ensure_ascii=False, indent=2))


MAPPING (corrigé):
 {
  "x": "EntityIdentifier",
  "y": "HasAmmunitionSupplyCap",
  "color": null,
  "size": null,
  "facet": null,
  "time": null,
  "filter": [
    {
      "column": "EntityIdentifier",
      "op": "==",
      "value": 1
    },
    {
      "column": "HasAmmunitionSupplyCap",
      "op": ">=",
      "value": 0
    }
  ],
  "chart": "bar",
  "reasoning_brief": "Comparing EntityIdentifier with HasAmmunitionSupplyCap values, filtering for positive supply caps."
}


In [18]:
# Génération du graphique à partir du mapping corrigé
from lida import Manager, llm

# Résumé du dataset pour LIDA
tmp_file = "tmp_lida.csv"
df_small = df[[col for col in mapping.values() if isinstance(col, str) and col in df.columns]].copy()
df_small.to_csv(tmp_file, index=False)

lida_mgr = Manager(text_gen=llm("openai", model="qwen2.5:3b"))
summary = lida_mgr.summarize(tmp_file)

goal = build_goal_from_mapping(prompt, mapping)
charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")

if not charts:
    raise RuntimeError("No charts returned by LIDA.")

save_chart_png(charts[0], Path("out2.png"))
print("Graphique généré et sauvegardé sous out.png")


```python
import matplotlib.pyplot as plt
import pandas as pd

<imports>

def plot(data: pd.DataFrame):
    filtered_data = data[(data['EntityIdentifier'] == 'VRF262162:1') & (data['HasAmmunitionSupplyCap'] >= 0)]
    ax = filtered_data.plot(kind='bar', x='EntityIdentifier', y='HasAmmunitionSupplyCap')
    plt.title('Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l\'axe x et HasAmmunitionSupplyCap pour l\'axe y.', wrap=True)
    ax.legend(['HasAmmunitionSupplyCap'])
    return plt;

chart = plot(data) # data already contains the data to be plotted. Always include this line.
```
****
 no numeric data to plot


RuntimeError: No charts returned by LIDA.

In [ ]:
# Version améliorée de run_lida avec correction automatique du mapping

def run_lida_with_mapping_fix(
    file: str,
    prompt: str,
    out: str = "out.png",
    sample_rows: int = 5000,
    schema_cols: int = 120,
    n_goals: int = 6,
):
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Missing OPENAI_API_KEY env var")

    df = load_any_df(file, sample_rows=sample_rows)
    if df.empty:
        raise RuntimeError("Empty dataframe (parsed but no rows).")
    df = coerce_basic_types(df)

 
    schema = summarize_columns(df, max_cols=schema_cols)

    mapping = llm_map_columns(prompt, schema)
    mapping = sanitize_mapping(mapping, df)
    mapping = correct_mapping_with_prompt(mapping, schema, prompt)
    print("MAPPING (corrigé):\n", json.dumps(mapping, ensure_ascii=False, indent=2))

    df2 = apply_filters(df, mapping)
    df_small = reduce_df_to_mapping(df2, mapping)

    # CSV temporaire pour LIDA
    import tempfile
    from pathlib import Path

    with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False, encoding="utf-8", newline="") as tmp:
        tmp_path = Path(tmp.name)
        df_small.to_csv(tmp, index=False)

    try:
        lida_mgr = Manager(text_gen=llm("openai", model="qwen2.5:3b"))
        summary = lida_mgr.summarize(str(tmp_path))

        goal = build_goal_from_mapping(prompt, mapping)
        charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")

        if not charts:
            raise RuntimeError("No charts returned by LIDA.")

        save_chart_png(charts[0], Path(out))

        code = getattr(charts[0], "code", None)
        if isinstance(code, str) and code.strip():
            Path(out).with_suffix(".generated.py").write_text(code, encoding="utf-8")

        print(f" Saved: {out}")

    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass

# Exemple d'appel
run_lida_with_mapping_fix(
    file="data/lidata.log",
    prompt="Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l'axe x et HasAmmunitionSupplyCap pour l'axe y.",
    out="out.png",
    schema_cols=25
)


MAPPING (corrigé):
 {
  "x": "EntityIdentifier",
  "y": "HasAmmunitionSupplyCap",
  "color": null,
  "size": null,
  "facet": null,
  "time": null,
  "filter": [
    {
      "column": "EntityIdentifier",
      "op": "==",
      "value": 1
    },
    {
      "column": "HasAmmunitionSupplyCap",
      "op": ">=",
      "value": 0
    }
  ],
  "chart": "bar",
  "reasoning_brief": "Comparing EntityIdentifier with HasAmmunitionSupplyCap values, filtering out non-zero ammunition supply capacities."
}
```python
import matplotlib.pyplot as plt
import pandas as pd

<imports>

def plot(data: pd.DataFrame):
    filtered_data = data[(data['EntityIdentifier'] == 1) & (data['HasAmmunitionSupplyCap'] >= 0)]
    ax = filtered_data.plot(kind='bar', x='EntityIdentifier', y='HasAmmunitionSupplyCap')
    plt.title('Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l'axe x et HasAmmunitionSupplyCap pour l'axe y.', wrap=True)
    ax.set_

RuntimeError: No charts returned by LIDA.

In [20]:
# Variante robuste de run_lida avec correction du mapping et affichages de debug


def run_lida_debug(
    file: str,
    prompt: str,
    out: str = "out.png",
    sample_rows: int = 5000,
    schema_cols: int = 120,
    n_goals: int = 6,
):
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Missing OPENAI_API_KEY env var")


    df = load_any_df(file, sample_rows=sample_rows)
    if df.empty:
        raise RuntimeError("Empty dataframe (parsed but no rows).")
    df = coerce_basic_types(df)


    # 1) Schema
    schema = summarize_columns(df, max_cols=schema_cols)


    # 2) Prompt → colonnes (mapping corrigé)
    mapping = llm_map_columns(prompt, schema)
    mapping = sanitize_mapping(mapping, df)
    mapping = correct_mapping_with_prompt(mapping, schema, prompt)
    mapping['filter'] = []  # Correction: retire tous les filtres pour éviter les erreurs de syntaxe
    print("\n--- MAPPING FINAL ---\n", json.dumps(mapping, ensure_ascii=False, indent=2))


    # 3) Filtrage + réduction
    df2 = apply_filters(df, mapping)
    df_small = reduce_df_to_mapping(df2, mapping)
    print("\n--- DF_SMALL HEAD ---\n", df_small.head())
    print("\n--- DF_SMALL COLUMNS ---\n", df_small.columns.tolist())


    # 4) CSV temporaire pour LIDA
    import tempfile
    from pathlib import Path


    with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False, encoding="utf-8", newline="") as tmp:
        tmp_path = Path(tmp.name)
        df_small.to_csv(tmp, index=False)


    try:
        lida_mgr = Manager(text_gen=llm("openai", model="qwen2.5:3b"))
        summary = lida_mgr.summarize(str(tmp_path))
        print("\n--- SUMMARY ---\n", summary)


        goal = build_goal_from_mapping(prompt, mapping)
        print("\n--- GOAL ---\n", json.dumps(goal, ensure_ascii=False, indent=2))
        charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")


        if not charts:
            print("\nAucun graphique généré par LIDA. Vérifiez le mapping, le goal et le résumé ci-dessus.")
            return


        save_chart_png(charts[0], Path(out))


        code = getattr(charts[0], "code", None)
        if isinstance(code, str) and code.strip():
            Path(out).with_suffix(".generated.py").write_text(code, encoding="utf-8")


        print(f"\nGraphique généré et sauvegardé sous {out}")


    finally:
        try:
            tmp_path.unlink(missing_ok=True)
        except Exception:
            pass


# Exemple d'appel
run_lida_debug(
    file="data/lidata.log",
    prompt="Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l'axe x et HasAmmunitionSupplyCap pour l'axe y.",
    out="out.png",
    schema_cols=25
)




--- MAPPING FINAL ---
 {
  "x": "EntityIdentifier",
  "y": "HasAmmunitionSupplyCap",
  "color": null,
  "size": null,
  "facet": null,
  "time": null,
  "filter": [
    {
      "column": "EntityIdentifier",
      "op": "==",
      "value": 1
    },
    {
      "column": "HasAmmunitionSupplyCap",
      "op": ">=",
      "value": 0
    }
  ],
  "chart": "bar",
  "reasoning_brief": "Comparing EntityIdentifier with HasAmmunitionSupplyCap values, filtering for positive supply caps."
}

--- DF_SMALL HEAD ---
 Empty DataFrame
Columns: [EntityIdentifier, HasAmmunitionSupplyCap]
Index: []

--- DF_SMALL COLUMNS ---
 ['EntityIdentifier', 'HasAmmunitionSupplyCap']

--- SUMMARY ---
 {'name': 'C:\\Users\\RAPHA\\AppData\\Local\\Temp\\tmppwzmn540.csv', 'file_name': 'C:\\Users\\RAPHA\\AppData\\Local\\Temp\\tmppwzmn540.csv', 'dataset_description': '', 'fields': [{'column': 'EntityIdentifier', 'properties': {'dtype': 'date', 'min': nan, 'max': nan, 'samples': [], 'num_unique_values': 0, 'semantic_type':

In [1]:
# Debug approfondi : affichage du mapping AVANT et APRES la suppression des filtres
file = "data/lidata.log"
prompt = "Trace un bar chart des EntityIdentifier qui ont des HasAmmunitionSupplyCap, en utilisant la colonne EntityIdentifier pour l'axe x et HasAmmunitionSupplyCap pour l'axe y."
schema_cols = 25
sample_rows = 5000

df = load_any_df(file, sample_rows=sample_rows)
df = coerce_basic_types(df)
schema = summarize_columns(df, max_cols=schema_cols)
mapping = llm_map_columns(prompt, schema)
mapping = sanitize_mapping(mapping, df)
mapping = correct_mapping_with_prompt(mapping, schema, prompt)
print("MAPPING AVANT SUPPRESSION FILTRES:\n", json.dumps(mapping, ensure_ascii=False, indent=2))
mapping['filter'] = []
print("MAPPING APRES SUPPRESSION FILTRES:\n", json.dumps(mapping, ensure_ascii=False, indent=2))

# On continue manuellement le process LIDA pour voir si le bug persiste
from lida import Manager, llm as lida_llm
import tempfile
from pathlib import Path

df2 = apply_filters(df, mapping)
df_small = reduce_df_to_mapping(df2, mapping)
print("\n--- DF_SMALL HEAD ---\n", df_small.head())
print("\n--- DF_SMALL COLUMNS ---\n", df_small.columns.tolist())

with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False, encoding="utf-8", newline="") as tmp:
    tmp_path = Path(tmp.name)
    df_small.to_csv(tmp, index=False)

try:
    lida_mgr = Manager(text_gen=lida_llm("openai", model="qwen2.5:3b"))
    summary = lida_mgr.summarize(str(tmp_path))
    print("\n--- SUMMARY ---\n", summary)
    goal = build_goal_from_mapping(prompt, mapping)
    print("\n--- GOAL ---\n", json.dumps(goal, ensure_ascii=False, indent=2))
    charts = lida_mgr.visualize(summary=summary, goal=goal, library="matplotlib")
    if not charts:
        print("\nAucun graphique généré par LIDA. Vérifiez le mapping, le goal et le résumé ci-dessus.")
    else:
        save_chart_png(charts[0], Path("out_debug.png"))
        print("\nGraphique généré et sauvegardé sous out_debug.png")
finally:
    try:
        tmp_path.unlink(missing_ok=True)
    except Exception:
        pass


NameError: name 'load_any_df' is not defined